In [68]:
import pandas as pd
import numpy as np
from collections import Counter
from nltk.util import bigrams
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from collections import Counter
import scipy
from scipy.special import softmax
import utils
import loglin
from sklearn.metrics import accuracy_score

In [30]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/mteb/imdb/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/mteb/imdb/" + splits["test"])

In [31]:
print(df_test.head())

                                                text  label label_text
0  I love sci-fi and am willing to put up with a ...      0   negative
1  Worth the entertainment value of a rental, esp...      0   negative
2  its a totally average film with a few semi-alr...      0   negative
3  STAR RATING: ***** Saturday Night **** Friday ...      0   negative
4  First off let me say, If you haven't enjoyed a...      0   negative


In [32]:
print(df_train.head())

                                                text  label label_text
0  I rented I AM CURIOUS-YELLOW from my video sto...      0   negative
1  "I Am Curious: Yellow" is a risible and preten...      0   negative
2  If only to avoid making this type of film in t...      0   negative
3  This film was probably inspired by Godard's Ma...      0   negative
4  Oh, brother...after hearing about this ridicul...      0   negative


In [33]:
print("Train label distribution:")
print(df_train['label'].value_counts())
print("Total amount of train entries:")
print(len(df_train))
print("Test label distribution:")
print(df_test['label'].value_counts())
print("Total amount of test entries:")
print(len(df_test))

Train label distribution:
label
1    12472
0    12432
Name: count, dtype: int64
Total amount of train entries:
24904
Test label distribution:
label
1    12412
0    12266
Name: count, dtype: int64
Total amount of test entries:
24678


The dataset is divided into a train set of 24904 rows and a test set of 24678 rows. Each row contains four columns: the review text, a label (0 for negative, 1 for positive), a label_text (negative or positive) and split (train or test). Both sets are almost equally balanced: the training set consists of 12472 positive (50.08%) and 12432 negative (50.02%) reviews, while the test set contains 12412 positive (50.30%) and 12266 negative (49.70%) reviews. Because the classes are evenly distributed, accuracy can be used as an evaluation metric for this analysis.

In [34]:
# I used the same parameteres as the tutorial used in the AG news example
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Metaspace()
trainer = BpeTrainer(vocab_size=1000, min_frequency=2, special_tokens=["[UNK]"])
corpus = df_train['text'].tolist()
tokenizer.train_from_iterator(corpus, trainer)

In [35]:
sample_text = df_test["text"].iloc[0]
output = tokenizer.encode(sample_text)
print(sample_text)
print(output.tokens)

I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn't match the background, and painfully one-dimensional characters cannot be overcome with a 'sci-fi' setting. (I'm sure there are those of you out there who think Babylon 5 is good sci-fi TV. It's not. It's clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It's really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it's rubbish as they have to alway

In [36]:
class TokenIndicator:

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding 
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

In [37]:
class BagOfWords:

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        token_ids = self.tokenizer.encode(x).ids
        counts = Counter(token_ids)

        vector = np.zeros(len(self))
        for token_id, count in counts.items():
            vector[token_id] = float(count)
        return vector

In [75]:
class BagOfBigrams:

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        self.vocab_size = tokenizer.get_vocab_size()
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())**2

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        token_ids = self.tokenizer.encode(x).ids
        bi_grams = bigrams(token_ids)

        bigram_ids = [id1*self.vocab_size + id2 for id1, id2 in bi_grams]
        return Counter(bigram_ids)

In [39]:
sample_text = df_test["text"].iloc[0]
indicator = TokenIndicator(tokenizer)
bow = BagOfWords(tokenizer)
bob = BagOfBigrams(tokenizer)
indicator_output = indicator(sample_text)
bow_output = bow(sample_text)
bob_output = bob(sample_text)
print(indicator_output.sum())
print(bow_output.sum())
print(bob_output.sum())
print(len(indicator_output))
print(len(bow_output))
print(len(bob_output))

261.0
513.0
512.0
1000
1000
1000000


In [82]:
train_data = list(zip(df_train['text'], df_train['label']))
encoders = [TokenIndicator, BagOfWords]
for enc in encoders:
    encoder = enc(tokenizer)
    logistic_model = loglin.fit(train_data, encoder, save_to=None)
    logistic_predictions = loglin.predict(df_test['text'].tolist(), logistic_model, encoder)
    accuracy = accuracy_score(df_test['label'].tolist(), logistic_predictions)
    print(enc.__name__)
    print(accuracy)

c:\Users\alwin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\alwin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


TokenIndicator
0.8200826647216144


c:\Users\alwin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\alwin\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


BagOfWords
0.8275792203582137
